# w21d_cpu2_jaad_eth_dut_imptc
Inputs: `w1-gpu2-dut-pose-and-graph` | Accelerator: `None` (Kaggle 4-vCPU Cloud)


In [ ]:
import hashlib, json, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Kaggle Cloud environment required'
REV = '26aef1d2665cea0491c0f7498c40884aca1421d3'
with tempfile.TemporaryDirectory(prefix='w21d_c2_') as ts:
    tmp = Path(ts)
    repo, sc0, sc1 = tmp / 'repo', tmp / 'sc0', tmp / 'sc1'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env2 = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2'}
    dut_hits = list(MOUNT.rglob('dut_pose_and_graph.zip'))
    dut_args = ['--dut-zip', str(dut_hits[0])] if dut_hits else []
    z0, z1 = tmp / 'jaad_eth.zip', tmp / 'dut_imptc.zip'
    p0 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave21_jaad_deocclusion_and_eth_ucy_6dataset',
        '--scratch-dir', str(sc0), '--only', 'jaad_eth', '--zip-path', str(z0)
    ], cwd=repo, env=env2)
    p1 = subprocess.Popen([
        sys.executable, '-u', '-m', 'scripts.run_wave21_jaad_deocclusion_and_eth_ucy_6dataset',
        '--scratch-dir', str(sc1), '--only', 'dut_imptc', *dut_args, '--imptc-tar', str(sc1 / 'imptc_sample.tar.zst'), '--zip-path', str(z1)
    ], cwd=repo, env=env2)
    rc0, rc1 = p0.wait(), p1.wait()
    assert rc0 == 0 and rc1 == 0, f'Subprocess failed: rc0={rc0}, rc1={rc1}'
    with zipfile.ZipFile(z0) as a0, zipfile.ZipFile(z1) as a1:
        s0, s1 = json.loads(a0.read('summary.json')), json.loads(a1.read('summary.json'))
    merged = {
        'experiment': 'wave21_jaad_deocclusion_and_eth_ucy_6dataset',
        'jaad2_human_deocclusion_346vid': s0['jaad2_human_deocclusion_346vid'],
        'eth_ucy_5scene_lodo': s0['eth_ucy_5scene_lodo'],
        'dut_3066_maneuver_onset': s1['dut_3066_maneuver_onset'],
        'imptc_564_maneuver_onset': s1['imptc_564_maneuver_onset'],
    }
    raw = (json.dumps(merged, indent=2, allow_nan=False) + '\n').encode('utf-8')
    sh = (json.dumps({'summary.json': hashlib.sha256(raw).hexdigest()}, indent=2) + '\n').encode('utf-8')
    out_z = WORK / 'wave21_jaad_deocclusion_and_eth_ucy_6dataset.zip'
    with zipfile.ZipFile(out_z, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('summary.json', raw); zf.writestr('sha256.json', sh)
    print('DONE:', out_z, out_z.stat().st_size, hashlib.sha256(out_z.read_bytes()).hexdigest(), flush=True)
